In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import geopandas as gpd
import pyogrio

PROJECT_DIR = Path.cwd().parent

PROCESSED = PROJECT_DIR / "data" / "processed"
ELEVATION = PROCESSED / "elevation"
QGIS_DIR = PROJECT_DIR / "qgis"

QGIS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT = PROCESSED / "where_roads_end.gpkg"

ADMIN_FILE = PROCESSED / "administrative_boundaries.gpkg"
ROADS_FILE = PROCESSED / "roads_classified.gpkg"

print("Project:", PROJECT_DIR.resolve())
print("Output:", OUTPUT.resolve())

Project: C:\Users\m.maxsudov\30daymapchallenge-2026\projects\day02_lines
Output: C:\Users\m.maxsudov\30daymapchallenge-2026\projects\day02_lines\qgis\where_roads_end.gpkg


In [2]:
districts = gpd.read_file(
    ADMIN_FILE, layer="districts"
).to_crs(4326)

regions = gpd.read_file(
    ADMIN_FILE, layer="regions"
).to_crs(4326)

country = gpd.read_file(
    ADMIN_FILE, layer="country"
).to_crs(4326)

roads = gpd.read_file(
    ROADS_FILE, layer="roads_classified"
).to_crs(4326)

stats = pd.read_csv(
    ELEVATION / "district_elevation_stats.csv",
    dtype={"district_id": "string", "osm_id": "string"}
)

highest = gpd.read_file(
    ELEVATION / "highest_road_points.gpkg",
    layer="highest_points"
).to_crs(4326)

candidates = gpd.read_file(
    ELEVATION / "top_road_candidates.gpkg",
    layer="top_candidates"
).to_crs(4326)

print("Districts:", len(districts))
print("Regions:", len(regions))
print("Roads:", len(roads))
print("Statistics:", len(stats))
print("Highest points:", len(highest))
print("Candidates:", len(candidates))

Districts: 203
Regions: 14
Roads: 571899
Statistics: 609
Highest points: 598
Candidates: 5801


In [3]:
def clean_id(series):
    return (
        series.astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

districts["district_id"] = clean_id(districts["osm_id"])
stats["district_id"] = clean_id(stats["district_id"])

metric_names = {
    "asphalt": "max_asphalt_m",
    "conventional": "max_conventional_m",
    "all_drivable": "max_drivable_m",
}

assert not stats.duplicated(
    ["district_id", "category"]
).any(), "Duplicate district/category combinations"

wide = stats.pivot(
    index="district_id",
    columns="category",
    values="max_elevation_m"
).rename(columns=metric_names)

wide = wide.reindex(columns=list(metric_names.values()))
wide = wide.reset_index()

districts_map = districts.merge(
    wide,
    on="district_id",
    how="left",
    validate="one_to_one"
)

print("Districts after merge:", len(districts_map))
print(
    districts_map[
        list(metric_names.values())
    ].notna().sum()
)

Districts after merge: 203
max_asphalt_m         192
max_conventional_m    203
max_drivable_m        203
dtype: int64


In [4]:
districts_map["gap_asphalt_all_m"] = (
    districts_map["max_drivable_m"]
    - districts_map["max_asphalt_m"]
)

districts_map["gap_conventional_all_m"] = (
    districts_map["max_drivable_m"]
    - districts_map["max_conventional_m"]
)

for metric, rank_name in [
    ("max_asphalt_m", "rank_asphalt"),
    ("max_conventional_m", "rank_conventional"),
    ("max_drivable_m", "rank_drivable"),
]:
    districts_map[rank_name] = (
        districts_map[metric]
        .rank(method="min", ascending=False)
        .astype("Int64")
    )

display_columns = [
    "name",
    "max_asphalt_m",
    "max_conventional_m",
    "max_drivable_m",
    "gap_asphalt_all_m",
    "rank_drivable"
]

print(
    districts_map[display_columns]
    .sort_values("max_drivable_m", ascending=False)
    .head(15)
    .to_string(index=False)
)

             name  max_asphalt_m  max_conventional_m  max_drivable_m  gap_asphalt_all_m  rank_drivable
       Pop tumani     2151.18680           3080.2732       3420.1345         1268.94770              1
     Zomin Tumani     2446.57620           2490.8540       3234.3152          787.73900              2
Shahrisabz tumani      805.84094           2546.7593       3120.6575         2314.81656              3
 Ohangaron Tumani     1961.18230           2902.1130       2902.1130          940.93070              4
   Qamashi tumani     1547.28000           2737.6885       2880.7583         1333.47830              5
 Yakkabog' tumani      607.84717           1902.1138       2800.4045         2192.55733              6
    Boysun Tumani     1515.06790           2175.7850       2693.8875         1178.81960              7
     Kitob tumani     1665.97220           2447.0356       2691.9463         1025.97410              8
      Uzun tumani     1391.59960           2663.5776       2663.5776     

In [5]:
def bool_flag(series):
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )

road_fields = [
    "id", "highway", "surface",
    "is_confirmed_asphalt",
    "is_conventional",
    "is_access_restricted",
    "is_tunnel",
    "geometry"
]

available = [
    col for col in road_fields
    if col in roads.columns
]

roads_map = roads[available].copy()

for col in [
    "is_confirmed_asphalt",
    "is_conventional",
    "is_access_restricted",
    "is_tunnel",
]:
    if col not in roads_map.columns:
        roads_map[col] = False

    roads_map[col] = (
        bool_flag(roads_map[col]).astype("int8")
    )

roads_map["road_class"] = np.select(
    [
        roads_map["is_tunnel"].eq(1),
        roads_map["is_confirmed_asphalt"].eq(1),
        roads_map["is_conventional"].eq(1),
    ],
    [
        "tunnel",
        "asphalt",
        "conventional",
    ],
    default="track"
)

print(
    roads_map["road_class"]
    .value_counts()
    .to_string()
)

road_class
conventional    521822
track            35620
asphalt          14277
tunnel             180


In [6]:
category_labels = {
    "asphalt": "Confirmed asphalt",
    "conventional": "Conventional roads",
    "all_drivable": "All potentially drivable",
}

for layer in [highest, candidates]:
    layer["district_id"] = clean_id(layer["district_id"])
    layer["osm_id"] = clean_id(layer["osm_id"])

    layer["category_label"] = (
        layer["category"].map(category_labels)
    )

    layer["elevation_round_m"] = (
        layer["elevation_m"].round().astype("Int64")
    )

    layer["osm_url"] = (
        "https://www.openstreetmap.org/way/"
        + layer["osm_id"]
    )

    if "is_restricted" in layer.columns:
        layer["is_restricted"] = (
            bool_flag(layer["is_restricted"]).astype("int8")
        )

print(
    highest[
        [
            "district_name",
            "category",
            "elevation_round_m",
            "osm_url"
        ]
    ]
    .sort_values("elevation_round_m", ascending=False)
    .head(10)
    .to_string(index=False)
)

    district_name     category  elevation_round_m                                     osm_url
       Pop tumani all_drivable               3420 https://www.openstreetmap.org/way/266410404
     Zomin Tumani all_drivable               3234 https://www.openstreetmap.org/way/862527332
Shahrisabz tumani all_drivable               3121 https://www.openstreetmap.org/way/547984980
       Pop tumani conventional               3080 https://www.openstreetmap.org/way/512550512
 Ohangaron Tumani all_drivable               2902 https://www.openstreetmap.org/way/247173701
 Ohangaron Tumani conventional               2902 https://www.openstreetmap.org/way/247173701
   Qamashi tumani all_drivable               2881 https://www.openstreetmap.org/way/362951736
 Yakkabog' tumani all_drivable               2800 https://www.openstreetmap.org/way/362947354
   Qamashi tumani conventional               2738 https://www.openstreetmap.org/way/557210851
    Boysun Tumani all_drivable               2694 https://ww

In [8]:
assert len(districts_map) == 203
assert districts_map["district_id"].is_unique

assert (
    districts_map["max_drivable_m"]
    >= districts_map["max_conventional_m"]
).all()

mask = districts_map["max_asphalt_m"].notna()

assert (
    districts_map.loc[mask, "max_conventional_m"]
    >= districts_map.loc[mask, "max_asphalt_m"]
).all(), "Asphalt exceeds conventional road elevation"

assert districts_map[
    "max_asphalt_m"
].isna().sum() == 11

for name, layer in {
    "districts": districts_map,
    "roads": roads_map,
    "highest": highest,
    "candidates": candidates,
}.items():
    print(
        name,
        "| rows:", len(layer),
        "| invalid:", (~layer.geometry.is_valid).sum(),
        "| CRS:", layer.crs
    )

print("QA completed")

districts | rows: 203 | invalid: 0 | CRS: EPSG:4326
roads | rows: 571899 | invalid: 0 | CRS: EPSG:4326
highest | rows: 598 | invalid: 0 | CRS: EPSG:4326
candidates | rows: 5801 | invalid: 0 | CRS: EPSG:4326
QA completed


In [9]:
violations = districts_map.loc[
    districts_map["max_asphalt_m"].notna()
    & (
        districts_map["max_asphalt_m"]
        > districts_map["max_conventional_m"]
    ),
    [
        "name",
        "max_asphalt_m",
        "max_conventional_m",
        "max_drivable_m",
    ]
]

print("Actual violations:", len(violations))
print(violations.to_string(index=False))

Actual violations: 0
Empty DataFrame
Columns: [name, max_asphalt_m, max_conventional_m, max_drivable_m]
Index: []


In [10]:
assert len(districts_map) == 203
assert districts_map["district_id"].is_unique

# Both categories must be subsets of all_drivable
for metric in ["max_asphalt_m", "max_conventional_m"]:
    mask = districts_map[metric].notna()

    assert (
        districts_map.loc[mask, "max_drivable_m"]
        >= districts_map.loc[mask, metric]
    ).all(), f"{metric} exceeds all_drivable"

assert districts_map["max_asphalt_m"].isna().sum() == 11

print("QA completed successfully")

QA completed successfully


In [11]:
layers = {
    "country": country,
    "regions": regions,
    "districts": districts_map,
    "roads": roads_map,
    "highest_points": highest,
    "top_candidates": candidates,
}

if OUTPUT.exists():
    OUTPUT.unlink()

for name, gdf in layers.items():
    print(f"Exporting {name}: {len(gdf):,} features")

    export_gdf = gdf.copy()

    # SQLite/GeoPackage handles integer flags consistently
    for col in export_gdf.columns:
        if str(export_gdf[col].dtype) == "boolean":
            export_gdf[col] = export_gdf[col].astype("Int8")

    export_gdf.to_file(
        OUTPUT,
        layer=name,
        driver="GPKG",
        engine="pyogrio"
    )

print("\nExport completed:", OUTPUT.resolve())
print("Size (MB):", round(OUTPUT.stat().st_size / 1e6, 2))

print("\nLayers:")
print(pyogrio.list_layers(OUTPUT))

Exporting country: 1 features
Exporting regions: 14 features
Exporting districts: 203 features
Exporting roads: 571,899 features
Exporting highest_points: 598 features
Exporting top_candidates: 5,801 features

Export completed: C:\Users\m.maxsudov\30daymapchallenge-2026\projects\day02_lines\qgis\where_roads_end.gpkg
Size (MB): 183.33

Layers:
[['country' 'MultiPolygon']
 ['regions' 'MultiPolygon']
 ['districts' 'MultiPolygon']
 ['roads' 'LineString']
 ['highest_points' 'Point']
 ['top_candidates' 'Point']]
